In [ ]:
import pandas as pd

dataset = pd.read_csv("Retail_Transactions_Dataset.csv")

dataset.head()

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000000,2022-01-21 06:27:29,Stacey Price,"['Ketchup', 'Shaving Cream', 'Light Bulbs']",3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN
1,1000000001,2023-03-01 13:01:21,Michelle Carlson,"['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'P...",2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One)
2,1000000002,2024-03-21 15:37:04,Lisa Graves,['Spinach'],6,41.49,Credit Card,Houston,Department Store,True,Professional,Winter,NaN
3,1000000003,2020-10-31 09:59:47,Mrs. Patricia May,"['Tissues', 'Mustard']",1,39.34,Mobile Payment,Chicago,Pharmacy,True,Homemaker,Spring,NaN
4,1000000004,2020-12-10 00:59:59,Susan Mitchell,['Dish Soap'],10,16.42,Debit Card,Houston,Specialty Store,False,Young Adult,Winter,Discount on Selected Items


## EDA

### Entendendo os dados do dataset

In [3]:
## undestanding dataset
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Transaction_ID     1000000 non-null  int64  
 1   Date               1000000 non-null  str    
 2   Customer_Name      1000000 non-null  str    
 3   Product            1000000 non-null  str    
 4   Total_Items        1000000 non-null  int64  
 5   Total_Cost         1000000 non-null  float64
 6   Payment_Method     1000000 non-null  str    
 7   City               1000000 non-null  str    
 8   Store_Type         1000000 non-null  str    
 9   Discount_Applied   1000000 non-null  bool   
 10  Customer_Category  1000000 non-null  str    
 11  Season             1000000 non-null  str    
 12  Promotion          666057 non-null   str    
dtypes: bool(1), float64(1), int64(2), str(9)
memory usage: 92.5 MB


### Quantidade de itens vendidos

In [4]:
## total values
total_values_sold = dataset["Total_Cost"].sum()

total_values_sold

np.float64(52455220.400000006)

### Valor total de vendas por store_type

In [5]:
store_type = dataset.groupby("Store_Type")["Total_Items"].sum()

store_type

Store_Type
Convenience Store    918049
Department Store     915635
Pharmacy             917729
Specialty Store      914950
Supermarket          915772
Warehouse Club       913806
Name: Total_Items, dtype: int64

### Quantos itens cada Store_type vendeu e quantos % representa do total de itens vendidos

In [6]:
## how much represents in percentage of total sold items
store_type = dataset.groupby("Store_Type")["Total_Items"].sum()

total_sold_items = dataset["Total_Items"].sum()

percentage = (store_type / total_sold_items) * 100

df_percentage = pd.DataFrame({
    "Total_Items": store_type,
    "Percentage": percentage
})

df_percentage

,Total_Items,Percentage
Store_Type,,
Convenience Store,918049,16.704128
Department Store,915635,16.660204
Pharmacy,917729,16.698305
Specialty Store,914950,16.647741
Supermarket,915772,16.662697
Warehouse Club,913806,16.626925


### Qual o total de custo por Store_type e qual o % do total de custos.

In [7]:
total_costs = dataset["Total_Cost"].sum()

total_costs_group = dataset.groupby("Store_Type")["Total_Cost"].sum()
percentage_costs = (total_costs_group / total_costs) * 100

df_costs_percentage = pd.DataFrame({
    "Total_Cost": total_costs_group,
    "Percentage": percentage_costs
})

df_costs_percentage


,Total_Cost,Percentage
Store_Type,,
Convenience Store,8731901.36,16.646392
Department Store,8731555.57,16.645732
Pharmacy,8766679.01,16.712691
Specialty Store,8701600.22,16.588626
Supermarket,8763455.21,16.706545
Warehouse Club,8760029.03,16.700014


### Qual método de pagamento é mais usado e qual é menos usado.

In [8]:
## most and less used payments
payments_methods = dataset["Payment_Method"].value_counts()

payments_methods

Payment_Method
Cash              250230
Debit Card        250074
Credit Card       249985
Mobile Payment    249711
Name: count, dtype: int64

### Qual o método escolhido para as compras mais caras.

In [9]:
## most used method for most expensive purchases
most_used_payment = dataset["Payment_Method"].value_counts()

most_used_payment

Payment_Method
Cash              250230
Debit Card        250074
Credit Card       249985
Mobile Payment    249711
Name: count, dtype: int64

### Qual o método escolhido para as compras mais caras.

In [10]:
# Cria intervalos de custo para saber qual método de pagamento é mais comum em cada faixa de custo
bins = [10, 40, 80, 100]
labels = ["10-40", "40-80", "80+"]

dataset["Cost_Range"] = pd.cut(dataset["Total_Cost"],
                               bins=bins,
                               labels=labels,
                               right=True)
dataset["Cost_Range"]

0         40-80
1         10-40
2         40-80
3         10-40
4         10-40
          ...  
999995    10-40
999996      80+
999997    40-80
999998    10-40
999999    40-80
Name: Cost_Range, Length: 1000000, dtype: category
Categories (3, str): ['10-40' < '40-80' < '80+']

In [11]:
# Encontra o método de pagamento mais comum em cada faixa
result = dataset.groupby("Cost_Range")["Payment_Method"].agg(
    lambda x: x.value_counts().idxmax())

result

Cost_Range
10-40     Debit Card
40-80           Cash
80+      Credit Card
Name: Payment_Method, dtype: str

### Qual a quantidade de vendas por hora.

In [12]:
# Quantidade de vendas por hora
new_date = pd.to_datetime(dataset["Date"])

hour_labels = [
    "00:00-01:00", "01:00-02:00", "02:00-03:00", "03:00-04:00", "04:00-05:00",
    "05:00-06:00", "06:00-07:00", "07:00-08:00", "08:00-09:00", "09:00-10:00",
    "10:00-11:00", "11:00-12:00", "12:00-13:00", "13:00-14:00", "14:00-15:00",
    "15:00-16:00", "16:00-17:00", "17:00-18:00", "18:00-19:00", "19:00-20:00",
    "20:00-21:00", "21:00-22:00", "22:00-23:00", "23:00-00:00"
]

## extraindo hora de cada item
new_date_hour_item = new_date.dt.hour

## agrupando por hora e contando o número de itens vendidos em cada hora
sales_per_hour = dataset.groupby(new_date_hour_item)["Total_Items"].count()

# Ajustando o índice para exibir os rótulos de hora
sales_per_hour.index = hour_labels

# Ordenando as vendas por hora em ordem decrescente
sales_per_hour = sales_per_hour.sort_values(ascending=False)

sales_per_hour

10:00-11:00    42021
08:00-09:00    41926
03:00-04:00    41917
21:00-22:00    41894
18:00-19:00    41843
09:00-10:00    41841
15:00-16:00    41819
13:00-14:00    41817
12:00-13:00    41790
04:00-05:00    41768
14:00-15:00    41739
23:00-00:00    41698
20:00-21:00    41674
17:00-18:00    41655
19:00-20:00    41646
02:00-03:00    41617
05:00-06:00    41574
07:00-08:00    41521
06:00-07:00    41508
16:00-17:00    41505
22:00-23:00    41433
00:00-01:00    41416
01:00-02:00    41224
11:00-12:00    41154
Name: Total_Items, dtype: int64

### Período de maior venda durante o dia (escolha minha adicionar mais um aspecto na análise exploratória)

In [13]:
## calcular período em que maior venda (manha, tarde, noite ou madrugada)
labels = ["Madrugada", "Manhã", "Tarde", "Noite"]
bins = [0, 6, 12, 18, 24]

intervals = pd.cut(new_date.dt.hour, bins=bins, labels=labels)

period = dataset.groupby(intervals)["Total_Items"].sum()

period = period.sort_values(ascending=False)

period

Date
Tarde        1377167
Manhã        1373556
Madrugada    1371198
Noite        1145781
Name: Total_Items, dtype: int64

### Qual a quantidade de vendas por dia da semana.

In [14]:
## converte para datetime
dataset_week = pd.to_datetime(dataset["Date"])
labels = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

## pega cada dia da semana [0, 1, 2, 3, 4, 5, 6]
days_of_week = dataset_week.dt.day_of_week

## transforma em uma serie com o index days_of_week e calcula a ocorrencia com count()
group_by_days_of_week = dataset.groupby(days_of_week)["Date"].count()

group_by_days_of_week.index = labels

group_by_days_of_week = group_by_days_of_week.sort_values(ascending=False)

group_by_days_of_week


Quinta     143375
Sexta      143367
Sábado     143310
Quarta     143104
Segunda    142435
Terça      142317
Domingo    142092
Name: Date, dtype: int64

### O dia e hora com mais venda por cidade.

In [15]:
date_column_to_datetime = pd.to_datetime(dataset["Date"])

date_column_day = date_column_to_datetime.dt.day
date_column_hour = date_column_to_datetime.dt.hour

city = dataset["City"]
sells = dataset["Total_Items"]

city_sells_date_dataframe = pd.DataFrame({
    "City": city,
    "Total_Items": sells,
    "Day": date_column_day,
    "Hour": date_column_hour
})

city_sells_date_dataframe = city_sells_date_dataframe.groupby(
    ["City", "Day", "Hour"], as_index=False)["Total_Items"].sum().sort_values(
        ["Total_Items", "Day", "Hour"], ascending=False)

city_sells_date_dataframe.head(100)

,City,Day,Hour,Total_Items
5349,New York,6,21,1045
93,Atlanta,4,21,1018
1097,Boston,15,17,990
6852,Seattle,7,12,990
4581,Miami,5,21,989
...,...,...,...,...
1766,Chicago,12,14,915
3724,Los Angeles,1,4,915
1953,Chicago,20,9,914
6285,San Francisco,14,21,913
